<a href="https://colab.research.google.com/github/LucasEmanuelRios/PrimerRepositorioGit/blob/main/Pasar_PDF_OC_A_EXCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import pdfplumber
import pandas as pd
import re

# === CONFIGURACIÓN ===
# Carpeta donde están los PDF
carpeta_pdfs = r"/content/sample_data/OC"  # Cambiá esta ruta

# Archivo de salida Excel
salida_excel = r"/content/sample_data/Resultado/resultado.xlsx"

# === FUNCIÓN PARA PROCESAR UN SOLO PDF ===
def procesar_pdf(ruta_pdf):
    rows = []
    with pdfplumber.open(ruta_pdf) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if not text:
                continue
            for line in text.splitlines():
                # Detectar líneas de posiciones
                # Ejemplo: "00010   SOFTWARE FALTANTE BUNDLE 2025 30.12.2025 1 Servicio 0,01 1 0,01"
                if line.strip().startswith("000") or line.strip().startswith("001") or line.strip().startswith("002"):
                    partes = line.split()
                    # Detectar posición (primer campo)
                    pos = partes[0]
                    # Extract the rest of the description line
                    desc_line = line[len(pos):].strip()

                    # Use regex to find the pattern: Date Quantity U.M. Unit_Price Factor Total_Price
                    # Date pattern: dd.mm.yyyy
                    # Quantity pattern: digit(s)
                    # U.M. pattern: Servicio or other unit
                    # Price pattern: number with comma or dot as decimal separator
                    pattern = r"(\d{2}\.\d{2}\.\d{4})\s+(\d+)\s+(\w+)\s+([\d,\.]+)\s+(\d+)\s+([\d,\.]+)"
                    match = re.search(pattern, desc_line)

                    fecha = ""
                    cantidad = ""
                    um = ""
                    precio_unit = ""
                    factor = ""
                    precio_total = ""
                    material_desc = desc_line

                    if match:
                        fecha = match.group(1)
                        cantidad = match.group(2)
                        um = match.group(3)
                        precio_unit = match.group(4)
                        factor = match.group(5)
                        precio_total = match.group(6)
                        # Remove the extracted parts from the material description
                        material_desc = desc_line[:match.start()].strip()


                    rows.append([pos, material_desc, fecha, cantidad, um, precio_unit, factor, precio_total])
    return rows

# === PROCESAR TODOS LOS PDF DE LA CARPETA ===
todas_filas = []
for archivo in os.listdir(carpeta_pdfs):
    if archivo.lower().endswith(".pdf"):
        ruta_pdf = os.path.join(carpeta_pdfs, archivo)
        filas = procesar_pdf(ruta_pdf)
        todas_filas.extend(filas)

# === CREAR DATAFRAME Y EXPORTAR A EXCEL ===
df = pd.DataFrame(todas_filas, columns=[
    "Posición",
    "Material / Descripción",
    "Fecha Entrega",
    "Cantidad",
    "U.M.",
    "Precio Unitario (USD)",
    "Factor",
    "Precio Total (USD)"
])

df.to_excel(salida_excel, index=False)

print(f"Excel generado en: {salida_excel}")

Excel generado en: /content/sample_data/Resultado/resultado.xlsx


In [ ]:
pip install pdfplumber pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 29.4 MB/s eta 0:00:00
